# 02 — Extract Running Text

- **GRETIL**: strip the `# Header` section, strip verse-reference codes, keep running Sanskrit lines.
- **DCS**: read `# text = …` comment lines from CoNLL-U; one comment line = one sentence.
- Every line is tagged with `text_id` (work-level, stable) and `source`.
- Output: `data/interim/gretil_extracted.jsonl` and `data/interim/dcs_extracted.jsonl`.

In [1]:
import json, re
from pathlib import Path
from tqdm.auto import tqdm

BASE    = Path('/Users/sidharthbildikar/Desktop/code/llm-paninian-compression/sanskrit_corpus')
RAW     = BASE / 'data' / 'raw'
INTERIM = BASE / 'data' / 'interim'
INTERIM.mkdir(parents=True, exist_ok=True)

GRETIL_DIR  = RAW / 'gretil'
DCS_CONLLU  = RAW / 'dcs' / 'sanskrit' / 'dcs' / 'data' / 'conllu' / 'files'

print('GRETIL files:', len(list(GRETIL_DIR.glob('sa_*.txt'))))
print('DCS CoNLL-U files:', len(list(DCS_CONLLU.rglob('*.conllu'))))


GRETIL files: 781
DCS CoNLL-U files: 15900


## Helper: derive stable text_id from GRETIL filename

Strip `sa_` prefix, `.txt` suffix, and edition/volume markers (`-edXxx`, `-[0-9]+`).

In [2]:
# Edition/volume markers that do NOT distinguish works — strip them
_EDITION_RE = re.compile(
    r'(-ed[A-Z][^-]*)'
    r'|(-crit)'
    r'|(-[0-9]+$)',
    re.IGNORECASE
)

def gretil_text_id(path: Path) -> str:
    stem = path.stem          # e.g. sa_Rgveda-edAufrecht
    if stem.startswith('sa_'):
        stem = stem[3:]        # Rgveda-edAufrecht
    stem = _EDITION_RE.sub('', stem)  # Rgveda
    return f'gretil_{stem}'

# Smoke-test
tests = [
    ('sa_Rgveda-edAufrecht.txt', 'gretil_Rgveda'),
    ('sa_mahabharata.txt',       'gretil_mahabharata'),
    ('sa_bhagavadgita-crit.txt', 'gretil_bhagavadgita'),
]
for fname, expected in tests:
    got = gretil_text_id(Path(fname))
    status = 'OK' if got == expected else f'FAIL (got {got!r})'
    print(f'  {fname} -> {got!r}  [{status}]')

  sa_Rgveda-edAufrecht.txt -> 'gretil_Rgveda'  [OK]
  sa_mahabharata.txt -> 'gretil_mahabharata'  [OK]
  sa_bhagavadgita-crit.txt -> 'gretil_bhagavadgita'  [OK]


## GRETIL extraction

Each GRETIL plaintext file has:
```
# Header
  ...
# Text
  <actual Sanskrit text>
```
Within the text section:
- Strip verse-reference codes (e.g. `RV_1,001.01`, `BhG_18.01`, `MBh_12.1.1`)
- Strip empty or whitespace-only lines
- Keep lines that still have non-whitespace content after stripping

In [ ]:
# ── verse-reference stripping (unchanged) ─────────────────────────────────
# Ref code format: letters + underscore + alphanumeric/punctuation incl. *
# e.g. RV_1,001.01  sbca_2.66  valc_1.1  dhvk_1.5a  Aprp_*1
_REF_CODE = r'[A-Za-z][A-Za-z0-9]*(?:_[A-Za-z0-9,.;/*\-]+)+'
_SEP      = r'(?:\|{1,3}|\.{2,}|/{1,2})'

_ONLY_REF = re.compile(
    r'^\s*(?:' + _SEP + r'\s*)?\(?'
    + _REF_CODE
    + r'\)?\s*(?:' + _SEP + r'|:)?\s*$'
)
_TRAILING_REF_WS = re.compile(
    r'\s+[A-Z][A-Za-z0-9]*(?:_[A-Za-z0-9,.;/*\-]+)+\s*$'
)
_ANY_REF = re.compile(
    r'\s*' + _SEP + r'\s*\(?'
    + _REF_CODE
    + r'\)?\s*(?:' + _SEP + r')?\s*'
)
_TRAILING_SEP = re.compile(r'\s*(?:\|+|\.{2,}|/{1,2})\s*$')
_HEADER_LINE  = re.compile(r'^#{1,3}\s')

# ── NEW: pre-transliteration noise patterns ────────────────────────────────
# Mahābhāṣya paragraph refs: (pas_1) (p_1,2.21) (p_2,1.1.1)
# These contain letters that transliterate to valid SLP1 — must strip here,
# BEFORE the text reaches 03_normalize, where the transliterator can't tell
# them apart from Sanskrit.
_MBH_PARA = re.compile(r'\(p[a-z]*_[^)]*\)')

# Mahābhāṣya edition cross-references: ka_i,1.1-5  ro_ii,31
# The \b word-boundary + underscore ensures no Sanskrit word is ever matched.
_MBH_KA   = re.compile(r'\bka_\S+')
_MBH_RO   = re.compile(r'\bro_\S+')

# Segment counters embedded in every Mahābhāṣya clause: {1/10} {3/21}
_MBH_SEG  = re.compile(r'\{\d+/\d+\}')

# Inline page references in prose commentaries: (p.5) (p.1) (p. 2)
# These survive extraction and whitelist as "p." — strip the whole token.
_INLINE_PAGE = re.compile(r'\([pP]\.?\s*\d+[-\d]*\)')

# Footnote reference markers embedded in verse text: [1] [23]
_FOOTNOTE_MK = re.compile(r'\[\d+\]')

# ── line-level skip patterns ───────────────────────────────────────────────
# Page-separator lines produced by critical editions: ---- vaidya, p. 1 ----
_PAGE_SEP_LINE = re.compile(r'^-{5,}')

# Notes-section boundary markers: __________notes__________
_NOTES_SEP = re.compile(r'_{5,}notes_{5,}', re.IGNORECASE)

# Footnote content lines (start with [N]) — the note text, not the Sanskrit
_FOOTNOTE_LINE = re.compile(r'^\s*\[\d+\]')

# Trailing editorial em-dash breaks often left at end of sentences: text ---
_EMDASH_END = re.compile(r'-{2,}\s*$')


def extract_gretil_lines(path: Path):
    """Return (text_id, list_of_cleaned_lines, error) for one GRETIL file."""
    text_id = gretil_text_id(path)
    try:
        raw = path.read_text(encoding='utf-8', errors='replace')
    except Exception as exc:
        return text_id, [], str(exc)

    # Fast-forward to the text body (skip file header)
    text_start = re.search(r'^#{1,3}\s*[Tt]ext\s*$', raw, re.MULTILINE)
    if text_start:
        raw = raw[text_start.end():]

    cleaned = []
    for line in raw.splitlines():
        # ── Line-level skips ───────────────────────────────────────────────
        if _HEADER_LINE.match(line):    continue   # residual ## markers
        if _NOTES_SEP.search(line):     continue   # ____notes____ separator
        if _PAGE_SEP_LINE.match(line):  continue   # ---- vaidya, p.1 ----
        if _FOOTNOTE_LINE.match(line):  continue   # [1] note text

        # ── Pre-transliteration inline stripping ──────────────────────────
        # Strip Mahābhāṣya cross-reference blocks before anything else.
        # Order matters: para first (removes parens+underscore), then ka_/ro_.
        line = _MBH_PARA.sub('', line)     # (pas_1), (p_1,2.21)
        line = _MBH_KA.sub('', line)       # ka_i,1.1-5
        line = _MBH_RO.sub('', line)       # ro_i,1-4
        line = _MBH_SEG.sub('', line)      # {1/10}
        line = _INLINE_PAGE.sub('', line)  # (p.5)
        line = _FOOTNOTE_MK.sub('', line)  # [1] inline marker in verse
        line = _EMDASH_END.sub('', line)   # trailing ---

        # ── Existing verse-reference stripping ────────────────────────────
        line = _TRAILING_REF_WS.sub('', line)
        if _ONLY_REF.match(line):
            continue
        segments = _ANY_REF.split(line)
        for seg in segments:
            seg = _TRAILING_SEP.sub('', seg).strip()
            if seg and re.search(r'[A-Za-zऀ-ॿ]', seg):
                cleaned.append(seg)

    return text_id, cleaned, None


# ── smoke tests ────────────────────────────────────────────────────────────
_smoke = [
    # Existing verse-ref cases (must still pass)
    ('na BadrakamidaM nATA .. sbca_2.66 ..',
     ['na BadrakamidaM nATA']),
    ('viśveśvaraṃ ca viśvahetukam // valc_1.1 //',
     ['viśveśvaraṃ ca viśvahetukam']),
    ('kāvyasyātmā sa evārthas purā / (dhvk_1.5a)',
     ['kāvyasyātmā sa evārthas purā']),
    ('idam api ca || MBh_12.1.1',
     ['idam api ca']),

    # Mahābhāṣya inline refs — the main fix
    ('(pas_1) ka_i,1.1-5 ro_i,1-4 {1/10} atha śabdānuśāsanam .',
     ['atha śabdānuśāsanam .']),
    ('(p_1,2.21) ka_i,201.2-4 ro_ii,31 {1/4} iha kasmāt na bhavati .',
     ['iha kasmāt na bhavati .']),

    # Inline page refs in prose
    ('uktaṃ ca (p.5)',
     ['uktaṃ ca']),
    ('sa(p.1) evaṃ vicintayan',
     ['sa evaṃ vicintayan']),

    # Footnote markers within verse text
    ('puṇyajñānamayaṃ[1]pracitya vipulaṃ hetuṃ',
     ['puṇyajñānamayaṃ pracitya vipulaṃ hetuṃ']),

    # Page separator line → must be completely dropped
    ('-------------------- vaidya, p. 1 --------------------', []),
]

print('Smoke tests:')
all_ok = True
for inp, expected in _smoke:
    line = inp
    if (_NOTES_SEP.search(line) or _PAGE_SEP_LINE.match(line)
            or _FOOTNOTE_LINE.match(line) or _HEADER_LINE.match(line)):
        got = []
    else:
        line = _MBH_PARA.sub('', line)
        line = _MBH_KA.sub('', line)
        line = _MBH_RO.sub('', line)
        line = _MBH_SEG.sub('', line)
        line = _INLINE_PAGE.sub('', line)
        line = _FOOTNOTE_MK.sub('', line)
        line = _EMDASH_END.sub('', line)
        line = _TRAILING_REF_WS.sub('', line)
        if _ONLY_REF.match(line):
            got = []
        else:
            segs = _ANY_REF.split(line)
            got = []
            for s in segs:
                s = _TRAILING_SEP.sub('', s).strip()
                if s and re.search(r'[A-Za-zऀ-ॿ]', s):
                    got.append(s)
    ok = got == expected
    all_ok = all_ok and ok
    print(f'  {"OK" if ok else "FAIL"} | {inp[:62]!r}')
    print(f'        got:      {got}')
    if not ok:
        print(f'        expected: {expected}')
print('All OK\n' if all_ok else 'SOME FAILED\n')

# Quick sanity check on real files
rv_path  = GRETIL_DIR / 'sa_Rgveda-edAufrecht.txt'
mbh_path = GRETIL_DIR / 'sa_pataJjali-vyAkaraNamahAbhASya.txt'

if rv_path.exists():
    tid, lines, err = extract_gretil_lines(rv_path)
    print(f'Rigveda : {len(lines):,} lines  err={err}')
    for l in lines[:3]:
        print(f'  {repr(l[:90])}')

if mbh_path.exists():
    _, lines, _ = extract_gretil_lines(mbh_path)
    print(f'\nMahābhāṣya : {len(lines):,} lines')
    for l in lines[:3]:
        print(f'  {repr(l[:90])}')

In [4]:
# Process all GRETIL files
gretil_out = INTERIM / 'gretil_extracted.jsonl'
gretil_files = sorted(GRETIL_DIR.glob('sa_*.txt'))

gretil_stats = {'files': 0, 'lines': 0, 'errors': [], 'text_ids': set()}

with gretil_out.open('w', encoding='utf-8') as fout:
    for path in tqdm(gretil_files, desc='GRETIL extract'):
        text_id, lines, err = extract_gretil_lines(path)
        if err:
            gretil_stats['errors'].append((path.name, err))
            continue
        gretil_stats['files'] += 1
        gretil_stats['text_ids'].add(text_id)
        for line in lines:
            gretil_stats['lines'] += 1
            fout.write(json.dumps({'text_id': text_id, 'source': 'gretil',
                                   'file': path.name, 'text': line},
                                  ensure_ascii=False) + '\n')

print(f'GRETIL: {gretil_stats["files"]} files, '
      f'{gretil_stats["lines"]:,} lines, '
      f'{len(gretil_stats["text_ids"])} text_ids, '
      f'{len(gretil_stats["errors"])} errors')
if gretil_stats['errors']:
    print('Errors:', gretil_stats['errors'][:5])

GRETIL extract:   0%|          | 0/781 [00:00<?, ?it/s]

GRETIL: 781 files, 1,519,182 lines, 771 text_ids, 0 errors


## DCS extraction

Each CoNLL-U file has document-level comments (`## text: Work Name`) and sentence-level
comments (`# text = running text`). We use the `# text =` lines as the running text units.
Work directory name is the canonical `text_id`.

In [5]:
_SENT_TEXT  = re.compile(r'^# text = (.+)$')
_DOC_TEXT   = re.compile(r'^## text: (.+)$')

def extract_dcs_lines(path: Path, work_dir_name: str):
    """Return (text_id, list_of_sentence_texts) for one DCS CoNLL-U file."""
    # Use the work directory name (human-readable work title) as text_id
    text_id = f'dcs_{work_dir_name}'
    try:
        raw = path.read_text(encoding='utf-8', errors='replace')
    except Exception as exc:
        return text_id, [], str(exc)

    sentences = []
    for line in raw.splitlines():
        m = _SENT_TEXT.match(line)
        if m:
            sent = m.group(1).strip()
            if sent:
                sentences.append(sent)

    return text_id, sentences, None

# Test
sample_dir = DCS_CONLLU / 'Agnipurāṇa'
if sample_dir.exists():
    sample_file = next(sample_dir.glob('*.conllu'))
    tid, sents, err = extract_dcs_lines(sample_file, 'Agnipurāṇa')
    print(f'text_id: {tid}')
    print(f'Sentences: {len(sents)}')
    print('First 3:')
    for s in sents[:3]:
        print(' ', repr(s))

text_id: dcs_Agnipurāṇa
Sentences: 77
First 3:
  'agnir uvāca'
  'catuṣpādaṃ dhanurvedaṃ vede pañcavidhaṃ dvija'
  'rathanāgāśvapattīnāṃ yodhāṃścāśritya kīrtitaṃ'


In [6]:
# Process all DCS files
dcs_out = INTERIM / 'dcs_extracted.jsonl'
dcs_stats = {'files': 0, 'lines': 0, 'errors': [], 'text_ids': set()}

with dcs_out.open('w', encoding='utf-8') as fout:
    # Each subdirectory = one work
    for work_dir in tqdm(sorted(DCS_CONLLU.iterdir()), desc='DCS extract'):
        if not work_dir.is_dir():
            continue
        work_name = work_dir.name
        for path in sorted(work_dir.glob('*.conllu')):
            text_id, sents, err = extract_dcs_lines(path, work_name)
            if err:
                dcs_stats['errors'].append((path.name, err))
                continue
            dcs_stats['files'] += 1
            dcs_stats['text_ids'].add(text_id)
            for s in sents:
                dcs_stats['lines'] += 1
                fout.write(json.dumps({'text_id': text_id, 'source': 'dcs',
                                       'file': path.name, 'text': s},
                                      ensure_ascii=False) + '\n')

print(f'DCS: {dcs_stats["files"]} files, '
      f'{dcs_stats["lines"]:,} sentences, '
      f'{len(dcs_stats["text_ids"])} text_ids, '
      f'{len(dcs_stats["errors"])} errors')
if dcs_stats['errors']:
    print('Errors:', dcs_stats['errors'][:5])

DCS extract:   0%|          | 0/270 [00:00<?, ?it/s]

DCS: 15900 files, 754,726 sentences, 270 text_ids, 0 errors


## Summary

In [7]:
total_lines = gretil_stats['lines'] + dcs_stats['lines']
print('=== Extraction summary ===')
print(f'  GRETIL  : {gretil_stats["lines"]:>10,} lines   {len(gretil_stats["text_ids"]):>5} text_ids')
print(f'  DCS     : {dcs_stats["lines"]:>10,} lines   {len(dcs_stats["text_ids"]):>5} text_ids')
print(f'  TOTAL   : {total_lines:>10,} lines')
print(f'  Outputs : {gretil_out}')
print(f'            {dcs_out}')

=== Extraction summary ===
  GRETIL  :  1,519,182 lines     771 text_ids
  DCS     :    754,726 lines     270 text_ids
  TOTAL   :  2,273,908 lines
  Outputs : /Users/sidharthbildikar/Desktop/code/llm-paninian-compression/sanskrit_corpus/data/interim/gretil_extracted.jsonl
            /Users/sidharthbildikar/Desktop/code/llm-paninian-compression/sanskrit_corpus/data/interim/dcs_extracted.jsonl
